In [ ]:
from utils import json_pretty_print, setup_notebook

if not setup_notebook(required_keys=[]):
    raise ValueError("Failed to setup notebook, please check your .env file")

In [ ]:
from asyncio import sleep

from agent_platform.core.kernel_interfaces.thread_state import ThreadStateInterface
from agent_platform.core.streaming import StreamingDelta
from agent_platform.core.thread import ThreadMessage

In [ ]:
class RunInterfaceMock:
    @property
    def run_id(self):
        return "test-run-123"


class KernelInterfaceMock:
    @property
    def run(self):
        return RunInterfaceMock()


class ThreadStateInterfaceMock(ThreadStateInterface):
    def __init__(self, thread_id: str, agent_id: str):
        super().__init__(thread_id=thread_id, agent_id=agent_id)

    async def _send_delta_event(self, delta_object: StreamingDelta) -> None:
        json_pretty_print("### Delta", delta_object.model_dump())

    async def _commit_message_to_storage(self, message: ThreadMessage) -> None:
        json_pretty_print("### Message Committed", message.model_dump())


mock_thread_state = ThreadStateInterfaceMock(
    thread_id="test-thread-123",
    agent_id="my-testing-agent-456",
)
mock_thread_state.attach_kernel(KernelInterfaceMock())

In [ ]:
test_message = await mock_thread_state.new_agent_message()

test_message.agent_metadata["test"] = "test"
await test_message.stream_delta()
await sleep(0.8)

test_message.agent_metadata["test"] = "testing"
test_message.agent_metadata["test2"] = ["a", "b", "c"]
test_message.agent_metadata["test3"] = {"a": "1", "b": "2", "c": "3"}
await test_message.stream_delta()
await sleep(0.8)

test_message.new_thought("> Thinking through")
await test_message.stream_delta()
await sleep(0.8)

test_message.append_thought(" testing...")
await test_message.stream_delta()
await sleep(0.5)

test_message.append_content("This is")
await test_message.stream_delta()
await sleep(1.2)
test_message.append_content(" a test")
await test_message.stream_delta()
await sleep(1.5)
test_message.append_content(" message")
await test_message.stream_delta()
await sleep(0.6)
test_message.append_content("!")
await test_message.stream_delta()

await test_message.commit()